<a href="https://colab.research.google.com/github/yaswanthvuppala/NLP-RAG/blob/main/NLP/NLP_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implementation of Key-Word Retrieval

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
documents = [
"RAG retrieves relevant documents before answer generation.",
"Embeddings represent text as numerical vectors.",
"Python is useful for machine learning and data analysis.",
"Cosine similarity compares the direction of two vectors."
]

In [3]:
query = "How can I compare text vectors?"

In [4]:
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)

doc_vectors

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 30 stored elements and shape (4, 29)>

In [5]:
query_vector = vectorizer.transform([query])
query_vector

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2 stored elements and shape (1, 29)>

In [6]:
scores = cosine_similarity(query_vector,doc_vectors).flatten()
scores

array([0.        , 0.53708274, 0.        , 0.17681216])

In [7]:
ranked_indices =  np.argsort(scores) [::-1]

ranked_indices

array([1, 3, 2, 0])

In [8]:
for index in ranked_indices:
  print(f"Score: {scores[index]:.3f}")
  print(documents[index])
  print()


Score: 0.537
Embeddings represent text as numerical vectors.

Score: 0.177
Cosine similarity compares the direction of two vectors.

Score: 0.000
Python is useful for machine learning and data analysis.

Score: 0.000
RAG retrieves relevant documents before answer generation.



Implement Semantic Retrieval


In [9]:
from sentence_transformers import SentenceTransformer

In [10]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
doc_embeddings = model.encode(documents,normalize_embeddings=True)

In [12]:
query_embeddings = model.encode(query,normalize_embeddings=True)

In [13]:
scores = doc_embeddings @ query_embeddings

In [14]:
scores

array([0.21810982, 0.47397447, 0.15254012, 0.5741581 ], dtype=float32)

In [15]:
ranked_indices= np.argsort(scores)[::-1]

In [16]:
for index in ranked_indices:
  print(f"Score: {scores[index]:.3f}")
  print(documents[index])
  print()

Score: 0.574
Cosine similarity compares the direction of two vectors.

Score: 0.474
Embeddings represent text as numerical vectors.

Score: 0.218
RAG retrieves relevant documents before answer generation.

Score: 0.153
Python is useful for machine learning and data analysis.



Example for Vector Database


In [17]:
!pip install chromadb

In [18]:

import chromadb
from sentence_transformers import SentenceTransformer

# Load an embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create a local vector database
client = chromadb.Client()
collection = client.create_collection("college_notes")

# Documents to store
documents = [
    "TCP provides reliable and ordered data delivery.",
    "Python is a popular language for machine learning.",
    "SQL is used to query relational databases."
]

# Generate embeddings
embeddings = model.encode(documents).tolist()

# Store documents and their vectors
collection.add(
    ids=["1", "2", "3"],
    documents=documents,
    embeddings=embeddings
)

# Search using a natural-language question
query = "How does network communication ensure reliability?"
query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)

print(results["documents"])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[['TCP provides reliable and ordered data delivery.', 'SQL is used to query relational databases.']]


In [19]:
results

{'ids': [['1', '3']],
 'embeddings': None,
 'documents': [['TCP provides reliable and ordered data delivery.',
   'SQL is used to query relational databases.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.963908851146698, 1.8279974460601807]]}